# ML-07 — Baseline Action Score and Top-10 Review

This notebook builds a transparent, rule-based content-refresh queue. The score is decision support, not a causal model or forecast.


## 1. My rule and its reason codes

A page moves up the queue when it has visible demand and a practical opportunity: it is old, has a weak search position, has low CTR for its visibility, or is unusually short. The two signal checks are **visibility × freshness** and **visibility × search opportunity**.

Reason codes: `stale_visible_page`, `position_opportunity`, `low_ctr_visible_page`, `thin_visible_page`, and `general_review`. The score excludes `trend_direction`, `trend_pct`, `is_declining_label`, and identifiers; those are reserved for evaluation or grouping.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

candidate_paths = [
    Path('data/raw/content_refresh_anonymized.csv'),
    Path('../../data/raw/content_refresh_anonymized.csv'),
]
data_path = next((p for p in candidate_paths if p.exists()), None)
if data_path is None:
    data_path = 'https://raw.githubusercontent.com/iamuhd/my_ML_Intership_at_FLYRANK-AI/main/data/raw/content_refresh_anonymized.csv'

raw = pd.read_csv(data_path)
required = ['content_id', 'client_id', 'impressions_90d', 'content_age_days', 'avg_position', 'ctr', 'days_since_last_update', 'word_count', 'trend_direction']
missing = sorted(set(required) - set(raw.columns))
if missing:
    raise ValueError(f'Missing required columns: {missing}')

df = raw[(raw['impressions_90d'] > 0) & (raw['content_age_days'] >= 90)].drop_duplicates('content_id').copy()
for column in ['impressions_90d', 'avg_position', 'ctr', 'days_since_last_update', 'word_count', 'engagement_rate', 'sessions_90d']:
    if column in df.columns:
        df[column] = pd.to_numeric(df[column], errors='coerce').replace([np.inf, -np.inf], np.nan).fillna(0)

def percentile_rank(values):
    return pd.Series(values).rank(method='average', pct=True).fillna(0).to_numpy()

def minmax(values):
    values = pd.Series(values).astype(float).fillna(0)
    lo, hi = values.min(), values.max()
    return pd.Series(0.0, index=values.index) if hi == lo else (values - lo) / (hi - lo)

def reason_codes(row):
    reasons = []
    visible = row['impressions_90d'] >= 500
    if row['days_since_last_update'] >= 180 and visible:
        reasons.append('stale_visible_page')
    if visible and 0 < row['avg_position'] <= 20:
        reasons.append('position_opportunity')
    if visible and 0 < row['avg_position'] <= 20 and row['ctr'] < 0.5:
        reasons.append('low_ctr_visible_page')
    if 0 < row['word_count'] < 1200 and row['impressions_90d'] >= 250:
        reasons.append('thin_visible_page')
    return '|'.join(reasons) if reasons else 'general_review'

declining_rate = df['trend_direction'].eq('down').mean()
print(f'Rows screened: {len(df):,}')
print(f'Observed declining-label base rate: {declining_rate:.3f}')

## 2. Build the ranked queue (writes the CSV)

Percentile ranks reduce the effect of heavy-tailed traffic while keeping the rule readable.

In [ ]:
visibility = pd.Series(percentile_rank(np.log1p(df['impressions_90d'])), index=df.index)
freshness = pd.Series(percentile_rank(df['days_since_last_update']), index=df.index)
position_quality = (1 - minmax(df['avg_position'].clip(lower=1, upper=50))) * (df['avg_position'] > 0).astype(int)
position_opportunity = position_quality * visibility
low_ctr_opportunity = (1 - minmax(df['ctr'].clip(lower=0, upper=df['ctr'].quantile(0.99)))) * visibility
depth_gap = (1 - pd.Series(percentile_rank(df['word_count']), index=df.index)) * visibility

df['baseline_action_score'] = (0.35 * visibility + 0.25 * freshness + 0.20 * position_opportunity + 0.15 * low_ctr_opportunity + 0.05 * depth_gap).clip(0, 1)
df['reason_codes'] = df.apply(reason_codes, axis=1)

def action(code):
    codes = set(str(code).split('|'))
    if 'thin_visible_page' in codes:
        return 'expand_and_refresh'
    if 'low_ctr_visible_page' in codes or 'position_opportunity' in codes:
        return 'refresh_and_review_ctr'
    if 'stale_visible_page' in codes:
        return 'refresh'
    return 'monitor'

df['suggested_action'] = df['reason_codes'].map(action)
df['is_declining_label'] = df['trend_direction'].eq('down').astype(int)
df = df.sort_values(['baseline_action_score', 'impressions_90d'], ascending=[False, False]).reset_index(drop=True)
df['baseline_rank'] = np.arange(1, len(df) + 1)

output_columns = ['content_id', 'client_id', 'baseline_rank', 'baseline_action_score', 'suggested_action', 'reason_codes', 'is_declining_label', 'trend_direction', 'impressions_90d', 'clicks_90d', 'sessions_90d', 'avg_position', 'ctr', 'engagement_rate', 'content_age_days', 'days_since_last_update', 'word_count']
out = df[output_columns].copy()
output_dir = Path('work/outputs')
if not output_dir.exists() and Path('../../').exists():
    output_dir = Path('../../work/outputs')
output_dir.mkdir(parents=True, exist_ok=True)
output_path = output_dir / 'baseline_action_score.csv'
out.to_csv(output_path, index=False)

def precision_at_k(frame, k):
    return float(frame.head(min(k, len(frame)))['is_declining_label'].mean())

base_rate = float(out['is_declining_label'].mean())
print(f'Wrote {len(out):,} rows to {output_path}')
print(f'Base rate: {base_rate:.3f}')
print(f'Precision@10: {precision_at_k(out, 10):.3f}')
print(f'Precision@50: {precision_at_k(out, 50):.3f}')
print(out.head(3).to_string(index=False))

## 3. Top-10 review

For each item, show the action, reason code, confidence note, and what could make the recommendation wrong.

In [ ]:
top10 = out.head(10).copy()
def confidence(row):
    if row['impressions_90d'] >= 3000 and row['reason_codes'] != 'general_review':
        return 'medium-high: visible demand plus explicit review signals'
    if row['impressions_90d'] >= 500 and row['reason_codes'] != 'general_review':
        return 'medium: inspect page context before acting'
    return 'low-medium: directional because demand is limited'

def failure_mode(row):
    if row['avg_position'] == 0:
        return 'No valid position data; position reasoning is unavailable.'
    if row['word_count'] == 0:
        return 'Word count is missing or unmeasured; thin-content reasoning may be incomplete.'
    return 'Topic, intent, competition, or client mix may explain the association.'

review = top10[['baseline_rank', 'content_id', 'suggested_action', 'reason_codes', 'baseline_action_score', 'impressions_90d', 'avg_position', 'ctr', 'days_since_last_update', 'word_count']].copy()
review['confidence_note'] = top10.apply(confidence, axis=1).to_numpy()
review['what_could_be_wrong'] = top10.apply(failure_mode, axis=1).to_numpy()
print(review.to_string(index=False))

## 4. Weak picks + leakage check

Weak picks are high-ranked rows with limited evidence: a generic reason, low visibility, or no valid position. The score uses only current snapshot measurements. Trend fields and the derived label are used only after ranking for evaluation; IDs are not score inputs.

In [ ]:
top50 = out.head(50)
weak_mask = (top50['reason_codes'] == 'general_review') | (top50['impressions_90d'] < 250) | (top50['avg_position'] == 0)
weak_picks = top50.loc[weak_mask, ['baseline_rank', 'content_id', 'baseline_action_score', 'reason_codes', 'impressions_90d', 'avg_position', 'ctr', 'word_count']].head(10)
print('Weak-pick candidates:')
print(weak_picks.to_string(index=False) if len(weak_picks) else 'None found in the top 50.')

score_inputs = {'impressions_90d', 'avg_position', 'ctr', 'days_since_last_update', 'word_count', 'engagement_rate', 'sessions_90d'}
forbidden = {'trend_direction', 'trend_pct', 'is_declining_label', 'client_id', 'content_id'}
assert not (score_inputs & forbidden)
assert out['baseline_rank'].is_unique
print('Leakage check: PASS')
print('Forbidden fields excluded from score:', sorted(forbidden))

## 5. Self-check

- [x] Rule and reasoning are written before implementation.
- [x] Both signal checks are addressed.
- [x] Queue is ranked and written to `work/outputs/baseline_action_score.csv`.
- [x] Top 10 has action, reason code, confidence, and failure mode.
- [x] Weak picks and leakage are checked.
- [x] Base rate and Precision@K are reported.
- [x] Claims are observed, directional, and decision-support only.

In [ ]:
checks = {
    'data_loaded': len(df) > 0,
    'queue_written': output_path.exists(),
    'top_10_reviewed': len(review) == min(10, len(out)),
    'score_bounded': out['baseline_action_score'].between(0, 1).all(),
    'ranks_unique': out['baseline_rank'].is_unique,
    'leakage_check_passed': not (score_inputs & forbidden),
}
for name, passed in checks.items():
    print(f'{name}: {passed}')
assert all(checks.values())
print('Self-check complete.')